In [1]:
!pip install -e .
!pip install -r requirements.txt

Obtaining file:///C:/Users/DELL/OneDrive/Documents/timecost-gradient-machine
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for tcgm (pyproject.toml): started
  Building editable for tcgm (pyproject.toml): finished with status 'done'
  Created wheel for tcgm: filename=tcgm-0.1.0-0.editable-py3-none-any.whl size=2800 sha256=18424f3a00c97952c8873c2a8828c44643cfefe5d2649f1f86d3430b5b681c2b
  Stored in directory: C:\Users\DELL\AppData\Local\Temp\pip-ephem-wheel-cache-x1r5tssc\wheels\6f\51\a2\6a898796c298e028f7efe5434832e633393d3f

In [6]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import TimeSeriesSplit

# from .loss import grad_financial, financial_loss
# from .metrics import evaluate_financial_performance
# from .core import sigmoid, logit, TrainerEngine, detect_drift_series

# from tcgm import TimeCostGradientMachine, LeakageSafeTargetEncoder

from tcgm.loss import grad_financial
from tcgm.metrics import evaluate_financial_performance
from tcgm.core import sigmoid, logit, TrainerEngine, detect_drift_series
from tcgm.encoders import LeakageSafeTargetEncoder
from tcgm.cost import compute_expected_monetary_loss

In [7]:
# synthetic financial dataset (simple)
rng = np.random.RandomState(42)
n_rows = 10000
customer = rng.choice([f"cust_{i}" for i in range(2000)], size=n_rows)
start = pd.Timestamp("2022-01-01")
timestamps = [start + pd.Timedelta(days=int(x)) for x in rng.randint(0, 365, size=n_rows)]
merchant = rng.choice([f"merch_{i}" for i in range(500)], size=n_rows)
amount = np.exp(rng.normal(5, 1.0, size=n_rows))
exposure = np.maximum(1.0, amount * (0.2 + rng.rand(n_rows) * 1.5))

# base propensities
cust_r = {c: rng.beta(1, 40) for c in np.unique(customer)}
merch_r = {m: rng.beta(1, 20) for m in np.unique(merchant)}
base = np.array([cust_r[c] * 0.6 + merch_r[m] * 0.4 for c,m in zip(customer, merchant)])
prob = np.clip(base * (1 + (amount - amount.mean()) / (amount.std() * 6)), 1e-6, 0.2)
y = rng.binomial(1, prob)

df = pd.DataFrame({
    "customer_id": customer,
    "timestamp": timestamps,
    "merchant_id": merchant,
    "amount": amount,
    "exposure": exposure,
    "label": y
})
df.sort_values("timestamp", inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()

,customer_id,timestamp,merchant_id,amount,exposure,label
0,cust_1551,2022-01-01,merch_62,82.823381,116.298396,1
1,cust_1591,2022-01-01,merch_434,139.235028,121.925644,0
2,cust_912,2022-01-01,merch_378,66.280274,63.874417,1
3,cust_938,2022-01-01,merch_260,109.894729,57.744105,0
4,cust_1276,2022-01-01,merch_14,84.703742,52.245332,0


In [8]:
# leakage-safe encoding inside time window (demonstration: train/test split)
train_df, test_df = train_test_split(df, test_size=0.2, shuffle=False)  # keep order
encoder = LeakageSafeTargetEncoder(cols=["merchant_id"], prior=0.05)
encoder.fit_on_train(train_df, target_col="label")
tr_te = encoder.transform(train_df)
te_te = encoder.transform(test_df)

X_train = pd.DataFrame({
    "amount": train_df["amount"].values,
    "merchant_te": tr_te["merchant_id_te"].values,
    "timestamp": train_df["timestamp"].values
})
y_train = train_df["label"].values

X_test = pd.DataFrame({
    "amount": test_df["amount"].values,
    "merchant_te": te_te["merchant_id_te"].values,
    "timestamp": test_df["timestamp"].values
})
y_test = test_df["label"].values

NameError: name 'train_test_split' is not defined

In [ ]:
# train TCGM
tcgm = TimeCostGradientMachine(
    n_estimators=20,
    learning_rate=0.1,
    cost_fp=50.0,
    cost_fn=1000.0,
    time_col="timestamp",
    n_splits=4,
    random_state=42
)

tcgm.fit(X_train, y_train)

In [ ]:
# evaluate
report = tcgm.evaluate(X_test, y_test)
report

In [ ]:
# choose best monetary threshold on test set
probs = tcgm.predict_proba(X_test)[:,1]
from tcgm import compute_expected_monetary_loss
eml = compute_expected_monetary_loss(y_test, probs, test_df["exposure"].values, lgd=0.6, cost_fp=50.0)
print("Best threshold:", eml["best_threshold"])
print("Best monetary loss:", eml["best_loss"])

In [ ]:
# save model artifact
joblib.dump(tcgm, "tcgm_model.joblib")
print("Saved tcgm_model.joblib")